In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torch import tensor

import numpy as np
import pandas as pd
import networkx as nx

In [2]:
DATA_ROOT = '/home/alexanderb/LEMBAS-RNN-benchmark'

# Aims outline

Use same as first attempt but replace linear function with MML functon from LEMBAS

### Load datasets

In [3]:
net = pd.read_csv(f"{DATA_ROOT}/Full data files/network(full).tsv", sep='\t')

In [4]:
network = nx.from_pandas_edgelist(net, 
                             source='TF', 
                             target='Gene', 
                             edge_attr='Interaction')

In [5]:
gene_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/Geneexpression (full).tsv"), sep='\t', header=0)

In [6]:
TF_expressions = pd.read_csv((f"{DATA_ROOT}/Full data files/TF(full).tsv"), sep='\t', header=0)

In [8]:
#filter genes to nodes in network
TF_expressions = TF_expressions[[gene for gene in TF_expressions.columns if gene in list(network.nodes)]]
gene_expressions = gene_expressions[[gene for gene in gene_expressions.columns if gene in list(network.nodes)]] 

### Creating dataset object

In [9]:
#torch tutorial code to use accelerator when available
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [10]:
from torch.utils.data import Dataset


class CustomTFGE(Dataset):
    def __init__(self, device, TF_expressions, gene_expressions, target_gene, network, transform=None, target_transform=None):
        '''
        Custom dataset for loading expression data across all TFs and samples and one target gene expression across all samples
        For a given index will return all TF expressions in one sample and target gene expression across all samples

        Parameters
        --------------
        device : torch device
            torch device to put dataset on, must be the same device as the model
        TF_expressions : Pandas dataframe
            pandas dataframe of TF expressions where columns are gene names and rows are samples. Expressions should be TPM normalised and log10 transformed.
        gene_expressions : Pandas dataframe
            pandas dataframe of target gene expressions, columns are gene names and rows are samples. Expressions should be TPM normalised and log10 transformed.
        target_gene : String
            String containing name of target gene for given model.
        network : networkx network
            Undirected network of the singalling pathway of interest. Used to filter TFs used when predicting target gene expression.
        '''
        #no transforms needed so set to none
        self.transform = transform
        self.target_transform = target_transform

        #load network
        self.network = network
        self.TF_expressions = TF_expressions
        self.gene_expressions = gene_expressions

        #subset to just target gene of interest
        self.target_gene = target_gene
        self.gene_expressions = self.gene_expressions[self.target_gene]

        #find genes in path and filter TFs accordingly
        self.path_genes = self.genes_in_path(self.network, list(self.TF_expressions.columns), self.target_gene)
        self.TF_expressions = self.TF_expressions [[gene for gene in self.TF_expressions.columns if gene in list(self.path_genes)]]

        #convert to torch tensors
        self.TF_expressions = torch.tensor(np.asarray(self.TF_expressions).T, dtype = torch.float32, device = device)
        self.gene_expressions = torch.tensor(np.asarray(self.gene_expressions), dtype = torch.float32, device = device)

    def genes_in_path(self, graph, source_nodes, target_node):
        '''
        Paramaters
        -------------
        parameter : graph
            networkx graph
        parameter : soruce_nodes
            list of node labels you wish to use as source nodes
        target_node : string
            label of desired target node
        
        Returns
        -------------
        nodes : List
            List of node labels in the network that are present in at least on shortest path to the specified target node from any of the source nodes.
        '''
        #keep track of all unique nodes found in a shortest path
        node_set = set()
        for source_node in source_nodes:
            #catch errors where there a no path from source to target
            try:
                sps = list(nx.all_shortest_paths(graph, 
                                source=source_node, 
                                target=target_node,
                                weight = None))
            except nx.NetworkXNoPath:
                print(f'No path from {source_node} to {target_node}')

            for path_nodes in sps:
                node_set.update(set(path_nodes))

        return(list(node_set))

    def __len__(self):
        #length of the dataset is the number of samples (not TFs in the dataset) - 15935
        return self.TF_expressions.shape[1]

    def __getitem__(self, idx):
        #get all TFs from sample correspinding to index
        TFs_exp = self.TF_expressions[:, idx]
        #get the target gene expression for the target model
        Gene_exp = self.gene_expressions[idx]
        #returns TFs for sample idx and target gene for sample idx
        return TFs_exp, Gene_exp

In [11]:
#initialise an instance of the dataset object - pass device so tensors and model on same device
dataset = CustomTFGE(device, TF_expressions=TF_expressions, gene_expressions=gene_expressions, network = network, target_gene = 'AADAT')
dataset

No path from DBX1 to AADAT
No path from ZNF512B to AADAT


In [12]:
#new way to create train and test dataset with pytorches dataset objects
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.8, 0.2])

In [13]:
#defining the neural network
class BasicNeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        #self.flatten = nn.Flatten()
        self.linear_layer = nn.Sequential(
            #linear activation layer takes 1198 TFs per sample -> outputs a value for the target gene for the same sample
            nn.Linear(1195, 1)
        )
    
    def forward(self, x):
        #forward pass is simply the linear layer
        expressions = self.linear_layer(x)
        return(expressions)

    

In [14]:
#put model on same device as the tensors
model = BasicNeuralNetwork().to(device)
print(model)

BasicNeuralNetwork(
  (linear_layer): Sequential(
    (0): Linear(in_features=1195, out_features=1, bias=True)
  )
)


## Train test loop

In [15]:
def train_loop(dataloader, model, loss_fn, optimiser):
    losses = []
    size = len(dataloader.dataset)
    
    #put model in train mode
    model.train()
    for batch, (X, y) in enumerate(dataloader):     
        #zero the gradient for each batch
        optimiser.zero_grad()

        #create a prediction
        pred = model(X)
        
        #calculate prediction loss
        loss = loss_fn(pred, y)

        #backpropogate for given loss
        loss.backward()
        optimiser.step()

        
        loss = loss.item()
        losses.append(loss)

        if batch +1 == size:
            print(f'Batch {batch}\n----------------')
            print(f'Epoch Finished at batch {batch}')

        #print loss every 500 batches - this is effectively every 500th sample
        if batch % 500 == 0:
            print(f"loss: {round(loss, 10)}")
        
    return(losses)



In [ ]:
def test_loss_distribution(dataloader, model, loss_fn, optimiser):
    losses = []
    #put model in eval mode
    model.eval()
    for batch, (X, y) in enumerate(dataloader):     

        #create a prediction - this will be 1 value for given sample and target gene
        pred = model(X)
        
        #calculate prediction loss - this will be one loss value in a distribution across samples per target gene
        loss = loss_fn(pred, y)

        loss = loss.item()
        losses.append(loss)
        
    return(losses)

In [17]:
#intialise hyperparameters - batch size is number of samples so that each backprop is done with the entire dataset (gradient descent not stochastic gradient descent)
#dataset is small enough for this to be fine
#investigate hyperparam tuning later
learning_rate = 1e-3
#run 1 sample at a time, but run through each sample per training epoch
batch_size = 1
epochs = 100

#initialize MSE loss function - same as LEMBAS
loss_fn = nn.MSELoss()

#initialise same optimiser as LEMBAS
optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [18]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [19]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    losses = train_loop(train_dataloader, model, loss_fn, optimiser)


Epoch 1
-------------------------------
loss: 0.020307757


/home/alexanderb/miniconda3/envs/MML_ML/lib/python3.12/site-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


loss: 0.0001885732
loss: 0.001068008
loss: 0.0427167565
loss: 0.1882561743
loss: 0.1586992443
loss: 0.0709518045
loss: 0.0469649173
loss: 0.0115743428
loss: 0.0017139663
loss: 0.1542251408
loss: 0.1675624251
loss: 0.0518248975
loss: 0.0190447923
loss: 0.0529172271
loss: 0.0160533544
loss: 0.0011977801
loss: 0.0103198579
loss: 0.1369754821
loss: 0.1300171614
loss: 0.2958887517
loss: 0.0390481167
loss: 0.0989911407
loss: 0.1430324465
loss: 0.0165886804
loss: 0.0196191929
Batch 12747
----------------
Epoch Finished at batch 12747
Epoch 2
-------------------------------
loss: 0.183727324
loss: 0.0491732322
loss: 0.0147391176
loss: 0.3002053499
loss: 0.130666241
loss: 0.0030745133
loss: 0.0105444118
loss: 0.1283359826
loss: 0.0156134982
loss: 0.0005364261
loss: 0.0414516218
loss: 0.1329798847
loss: 0.5038284063
loss: 0.0010866247
loss: 0.4591456652
loss: 0.0726781338
loss: 0.0211581979
loss: 0.0072879423
loss: 0.1421141028
loss: 0.0107240379
loss: 0.3929983377
loss: 0.3562675416
loss: 0.132

In [ ]:
MSE_distrubution = test_loss_distribution(test_dataloader, model, loss_fn, optimiser)

[0.02730928733944893,
 0.07354539632797241,
 2.015051359194331e-05,
 0.021332997828722,
 0.06634809076786041,
 0.0016372327227145433,
 0.0453687459230423,
 0.002468048594892025,
 0.09678851813077927,
 0.012031990103423595,
 0.004134237300604582,
 0.03421742469072342,
 0.018562037497758865,
 0.01986101269721985,
 0.018639834597706795,
 1.0109287359227892e-05,
 0.011090916581451893,
 0.00834784284234047,
 0.2851388454437256,
 0.17313261330127716,
 0.06351999938488007,
 0.04791020229458809,
 0.00404234416782856,
 2.5663650035858154,
 0.002549146069213748,
 0.008583609946072102,
 0.03346918150782585,
 0.04908973351120949,
 0.00033573200926184654,
 0.13878707587718964,
 0.030656011775135994,
 0.11191265285015106,
 0.08769971877336502,
 0.0003278050571680069,
 0.05249423533678055,
 0.003920941613614559,
 0.32412534952163696,
 0.09159830957651138,
 0.01008357759565115,
 0.07711030542850494,
 0.027900386601686478,
 0.005232387222349644,
 0.01601359434425831,
 0.023677844554185867,
 0.074340231

# Saving model

In [21]:
torch.save(model, 'models/linear_with_network.pth')

In [22]:
#How to load for reference
model = torch.load('models/linear_with_network.pth', weights_only=False)